# Assignment Case B — Vendor Performance
## Working with PySpark DataFrames for Analytics
**Day 42 | Batch 13 | 03 Mei 2026**

---

| Field | Isi |
|-------|-----|
| **Nama** | *(isi nama kamu)* |
| **NIM** | *(isi NIM kamu)* |
| **Case** | Case B — Procurement & Vendor Performance |
| **Source** | Hive: `adventureworks.*` |
| **Target** | Hive: `adventureworks_curated.fact_vendor_performance` |

---

## ⚠️ Filter Penting

> **Kamu HANYA boleh menggunakan data dengan ShipMethod: `CARGO TRANSPORT 5` dan `OVERNIGHT J-FAST`.**
> Filter ini harus diterapkan sebelum agregasi.

## 🎯 Business Question

1. **Vendor mana yang paling on-time** dalam pengiriman via CARGO TRANSPORT 5 dan OVERNIGHT J-FAST?
2. **Apakah harga beli dari vendor sudah kompetitif** dibanding StandardCost produk?
3. **Vendor mana yang memiliki VendorScore tertinggi** secara keseluruhan?
4. **Korelasi** antara CreditRating vendor dan OnTimeRate?

## 📐 Formula VendorScore

```
VendorScore = (OnTimeRate * 0.6) + ((100 - ABS(AvgPriceVariance)) * 0.4)
```

- **OnTimeRate** = OnTimeOrders / TotalOrders * 100  (ShipDate <= DueDate)
- **AvgPriceVariance** = rata-rata (UnitPrice - StandardCost) per baris PO detail
- ⚠️ Exclude baris dengan ShipDate IS NULL dari kalkulasi OnTimeRate

## 📋 Kolom Target: `fact_vendor_performance`

| Kolom | Tipe | Keterangan |
|-------|------|------------|
| `vendor_id` | INT | ID vendor |
| `vendor_name` | STRING | Nama vendor |
| `credit_rating` | INT | Rating kredit (1-5) |
| `ship_method` | STRING | Metode pengiriman |
| `total_orders` | INT | Total PO |
| `on_time_orders` | INT | PO dengan ShipDate <= DueDate |
| `on_time_rate` | DECIMAL(5,2) | % ketepatan pengiriman |
| `avg_lead_time_days` | DECIMAL(5,1) | Rata-rata (ShipDate - OrderDate) |
| `avg_price_variance` | DECIMAL(10,2) | Rata-rata (UnitPrice - StandardCost) |
| `vendor_score` | DECIMAL(5,2) | Formula scorecard |
| `load_timestamp` | TIMESTAMP | Waktu load |


## 0. Persiapan: Buat Hive External Tables untuk Vendor Performance

Sebelum mulai analisis, kamu perlu menyiapkan tabel sumber sebagai
**Hive External Tables**. Pola tutorialnya sama dengan:
- `02_adventureworks_to_hdfs.ipynb` → extract PostgreSQL ke HDFS Parquet (dimensi + fakta dengan partisi)
- `03_hdfs_to_hive.ipynb` → buat External Tables di atas Parquet

**Tabel yang akan dibuat di Hive (`adventureworks.*`):**
| Hive Table | Sumber PostgreSQL | Catatan |
|---|---|---|
| `fact_purchase_orders` | `Purchasing.PurchaseOrderHeader` | partisi `order_year`, `order_month` |
| `fact_purchase_details` | `Purchasing.PurchaseOrderDetail` | tanpa partisi |
| `dim_vendor` | `Purchasing.Vendor` | |
| `dim_ship_method` | `Purchasing.ShipMethod` | |
| `dim_product` | `Production.Product` | |

> **Lewati bagian ini** jika tabel sudah ada di Hive.

### 0.1 Setup SparkSession JDBC (untuk Extract ke HDFS)

Mirip dengan `02_adventureworks_to_hdfs.ipynb`. SparkSession ini
**tanpa Hive support** — hanya untuk extract data dari PostgreSQL ke HDFS sebagai Parquet.

In [1]:
# TODO: setup environment & import
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--jars /home/jovyan/work/jars/postgresql-42.7.3.jar pyspark-shell'

from pyspark.sql import SparkSession
from pyspark.sql import functions as F_jdbc

# buat spark_jdbc tanpa Hive support
spark_jdbc = SparkSession.builder \
    .master('local[*]') \
    .appName('Vendor-to-HDFS') \
    .config('spark.sql.parquet.compression.codec', 'snappy') \
    .getOrCreate()

# konfigurasi JDBC PostgreSQL AdventureWorks
JDBC_URL   = 'jdbc:postgresql://postgres:5432/postgres'
JDBC_PROPS = {'user': 'postgres', 'password': 'My_password1', 'driver': 'org.postgresql.Driver'}
HDFS_BASE  = 'hdfs://namenode:9000/datalake/raw'

def read_pg(schema, table):
    return spark_jdbc.read.jdbc(
        url=JDBC_URL,
        table=f'"{schema}"."{table}"',
        properties=JDBC_PROPS
    )

print(f'spark_jdbc {spark_jdbc.version} ready')

spark_jdbc 3.5.0 ready


### 0.2 Extract Tabel Dimensi ke HDFS

Tabel dimensi tidak perlu di-partisi (ukurannya kecil).

In [2]:
# lengkapi list tabel dimensi
dim_tables = [
    ('Purchasing', 'Vendor'),
    ('Purchasing', 'ShipMethod'),
    ('Production', 'Product'),
    ('Purchasing', 'PurchaseOrderDetail'),
]

for schema, table in dim_tables:
    df = read_pg(schema, table)
    hdfs_path = f'{HDFS_BASE}/{schema}/{table}'
    df.write.mode('overwrite').parquet(hdfs_path)
    print(f'  {schema}.{table:30s} -> {df.count():6,} rows -> HDFS')

print('Tabel dimensi selesai disimpan!')

  Purchasing.Vendor                         ->    104 rows -> HDFS
  Purchasing.ShipMethod                     ->      5 rows -> HDFS
  Production.Product                        ->    504 rows -> HDFS
  Purchasing.PurchaseOrderDetail            ->  8,845 rows -> HDFS
Tabel dimensi selesai disimpan!


### 0.3 Extract Tabel Fakta dengan Partisi

`PurchaseOrderHeader` berukuran besar dan punya kolom `OrderDate`. Partisi
berdasarkan `order_year` dan `order_month` — sama polanya dengan
`SalesOrderHeader` di notebook `02`.

In [3]:
# read PurchaseOrderHeader dari PostgreSQL
df_poh = read_pg('Purchasing', 'PurchaseOrderHeader')

# tambahkan kolom partisi order_year & order_month dari OrderDate
df_poh_part = df_poh \
    .withColumn('order_year',  F_jdbc.year(F_jdbc.col('OrderDate'))) \
    .withColumn('order_month', F_jdbc.month(F_jdbc.col('OrderDate')))

po_path = f'{HDFS_BASE}/Purchasing/PurchaseOrderHeader'

# simpan ke HDFS dengan partisi (order_year, order_month)
df_poh_part.write.mode('overwrite') \
    .partitionBy('order_year', 'order_month') \
    .parquet(po_path)

print(f'  PurchaseOrderHeader -> {df_poh.count():,} rows -> HDFS (partisi order_year/order_month)')

  PurchaseOrderHeader -> 4,012 rows -> HDFS (partisi order_year/order_month)


### 0.4 Setup SparkSession dengan Hive Support

Mirip dengan `03_hdfs_to_hive.ipynb`. Tutup `spark_jdbc` dulu, lalu buat
SparkSession baru **dengan** Hive support untuk membuat External Tables.

In [4]:
# stop spark_jdbc agar tidak konflik resource
spark_jdbc.stop()

from pyspark.sql import SparkSession

# buat spark_hive dengan Hive support
spark_hive = SparkSession.builder \
    .master('local[*]') \
    .appName('Vendor-HiveSetup') \
    .config('spark.jars', '/home/jovyan/work/jars/postgresql-42.7.3.jar') \
    .config('hive.metastore.uris', 'thrift://hive-metastore:9083') \
    .config('spark.sql.warehouse.dir', 'hdfs://namenode:9000/user/hive/warehouse') \
    .enableHiveSupport() \
    .getOrCreate()

HDFS_BASE = 'hdfs://namenode:9000/datalake/raw'

# buat database adventureworks jika belum ada
spark_hive.sql('CREATE DATABASE IF NOT EXISTS adventureworks')
spark_hive.sql('USE adventureworks')
print(f'spark_hive {spark_hive.version} ready')
spark_hive.sql('SHOW DATABASES').show()

spark_hive 3.5.0 ready
+--------------+
|     namespace|
+--------------+
|adventureworks|
|       default|
+--------------+



### 0.5 Buat Hive External Tables — Tabel Dimensi

Tabel dimensi tanpa partisi → schema otomatis di-infer.

In [5]:
# Helper function — sama persis dengan pola di 03_hdfs_to_hive.ipynb
def create_external_table(table_name, hdfs_path):
    spark_hive.sql(f'DROP TABLE IF EXISTS adventureworks.{table_name}')
    spark_hive.sql(f"""
        CREATE EXTERNAL TABLE adventureworks.{table_name}
        USING PARQUET
        LOCATION '{hdfs_path}'
    """)
    count = spark_hive.sql(f'SELECT COUNT(*) FROM adventureworks.{table_name}').collect()[0][0]
    print(f'  {table_name:35s} -> {count:6,} rows')

# lengkapi mapping nama tabel Hive -> path HDFS
hive_dim_tables = [
    ('dim_vendor',          f'{HDFS_BASE}/Purchasing/Vendor'),
    ('dim_ship_method',     f'{HDFS_BASE}/Purchasing/ShipMethod'),
    ('dim_product',         f'{HDFS_BASE}/Production/Product'),
    ('fact_purchase_details', f'{HDFS_BASE}/Purchasing/PurchaseOrderDetail'),
]

for tname, path in hive_dim_tables:
    create_external_table(tname, path)

print('Tabel dimensi Hive selesai!')

  dim_vendor                          ->    104 rows
  dim_ship_method                     ->      5 rows
  dim_product                         ->    504 rows
  fact_purchase_details               ->  8,845 rows
Tabel dimensi Hive selesai!


### 0.6 Buat Hive External Table — Tabel Fakta dengan Partisi

Untuk tabel berpartisi, perlu deklarasi `PARTITIONED BY` di DDL
lalu jalankan `MSCK REPAIR TABLE` agar Hive mengenali partisinya.
Pola sama dengan `fact_sales_orders` di `03_hdfs_to_hive.ipynb`.

In [6]:
po_path = f'{HDFS_BASE}/Purchasing/PurchaseOrderHeader'

# drop dulu kalau ada
spark_hive.sql('DROP TABLE IF EXISTS adventureworks.fact_purchase_orders')

# buat External Table dengan skema eksplisit + PARTITIONED BY
spark_hive.sql(f"""
    CREATE EXTERNAL TABLE adventureworks.fact_purchase_orders (
        PurchaseOrderID    INT,
        RevisionNumber     SMALLINT,
        Status             SMALLINT,
        EmployeeID         INT,
        VendorID           INT,
        ShipMethodID       INT,
        OrderDate          TIMESTAMP,
        ShipDate           TIMESTAMP,
        DueDate            TIMESTAMP,
        SubTotal           DECIMAL(18,2),
        TaxAmt             DECIMAL(18,2),
        Freight            DECIMAL(18,2),
        TotalDue           DECIMAL(18,2),
        ModifiedDate       TIMESTAMP
    )
    PARTITIONED BY (order_year INT, order_month INT)
    STORED AS PARQUET
    LOCATION '{po_path}'
""")

# jalankan MSCK REPAIR TABLE agar partisi terdeteksi
spark_hive.sql('MSCK REPAIR TABLE adventureworks.fact_purchase_orders')

cnt = spark_hive.sql('SELECT COUNT(*) FROM adventureworks.fact_purchase_orders').collect()[0][0]
print(f'  fact_purchase_orders -> {cnt:,} rows')

spark_hive.sql('SHOW TABLES IN adventureworks').show(20)
spark_hive.stop()
print('Setup Hive External Tables selesai.')

  fact_purchase_orders -> 4,012 rows
+--------------+--------------------+-----------+
|     namespace|           tableName|isTemporary|
+--------------+--------------------+-----------+
|adventureworks|        dim_customer|      false|
|adventureworks|      dim_department|      false|
|adventureworks|        dim_employee|      false|
|adventureworks|          dim_person|      false|
|adventureworks|         dim_product|      false|
|adventureworks|dim_product_category|      false|
|adventureworks|  dim_product_subcat|      false|
|adventureworks|     dim_ship_method|      false|
|adventureworks|       dim_territory|      false|
|adventureworks|          dim_vendor|      false|
|adventureworks|fact_purchase_det...|      false|
|adventureworks|fact_purchase_orders|      false|
|adventureworks|   fact_sales_orders|      false|
+--------------+--------------------+-----------+

Setup Hive External Tables selesai.


## 0. Setup SparkSession

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('CaseB_VendorPerformance') \
    .config('hive.metastore.uris', 'thrift://hive-metastore:9083') \
    .config('spark.sql.warehouse.dir', 'hdfs://namenode:9000/user/hive/warehouse') \
    .enableHiveSupport() \
    .getOrCreate()

print(f'Spark {spark.version} ready')
spark.sql('SHOW TABLES IN adventureworks').show(20)

Spark 3.5.0 ready
+--------------+--------------------+-----------+
|     namespace|           tableName|isTemporary|
+--------------+--------------------+-----------+
|adventureworks|        dim_customer|      false|
|adventureworks|      dim_department|      false|
|adventureworks|        dim_employee|      false|
|adventureworks|          dim_person|      false|
|adventureworks|         dim_product|      false|
|adventureworks|dim_product_category|      false|
|adventureworks|  dim_product_subcat|      false|
|adventureworks|     dim_ship_method|      false|
|adventureworks|       dim_territory|      false|
|adventureworks|          dim_vendor|      false|
|adventureworks|fact_purchase_det...|      false|
|adventureworks|fact_purchase_orders|      false|
|adventureworks|   fact_sales_orders|      false|
+--------------+--------------------+-----------+



## 1. Load Data dari Hive External Tables

Semua tabel sumber sudah ada di Hive sebagai External Tables (dibuat di Bagian 0).
Baca via `spark.table('adventureworks.<nama_tabel>')`.

In [8]:
# Load semua tabel dari Hive
df_po_header  = spark.table('adventureworks.fact_purchase_orders')
df_po_detail  = spark.table('adventureworks.fact_purchase_details')
df_vendor     = spark.table('adventureworks.dim_vendor')
df_shipmethod = spark.table('adventureworks.dim_ship_method')
df_product    = spark.table('adventureworks.dim_product')

print('Row counts:')
for name, df in [
    ('fact_purchase_orders',  df_po_header),
    ('fact_purchase_details', df_po_detail),
    ('dim_vendor',            df_vendor),
    ('dim_ship_method',       df_shipmethod),
    ('dim_product',           df_product),
]:
    print(f'  {name}: {df.count():,}')

Row counts:
  fact_purchase_orders: 4,012
  fact_purchase_details: 8,845
  dim_vendor: 104
  dim_ship_method: 5
  dim_product: 504


## 2. Eksplorasi Data

In [9]:
# Eksplorasi
print('=== ShipMethod yang tersedia ===')
df_shipmethod.select('Name').distinct().show(truncate=False)

print('\n=== Sample PurchaseOrderHeader ===')
df_po_header.show(5, truncate=False)

print('\n=== Distribusi PO per ShipMethod ===')
df_po_header.join(df_shipmethod, df_po_header.ShipMethodID == df_shipmethod.ShipMethodID) \
    .groupBy(df_shipmethod.Name.alias('ShipMethod')) \
    .count() \
    .show(truncate=False)

=== ShipMethod yang tersedia ===
+------------------+
|Name              |
+------------------+
|OVERSEAS - DELUXE |
|CARGO TRANSPORT 5 |
|XRQ - TRUCK GROUND|
|OVERNIGHT J-FAST  |
|ZY - EXPRESS      |
+------------------+


=== Sample PurchaseOrderHeader ===


Py4JJavaError: An error occurred while calling o203.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 18.0 failed 1 times, most recent failure: Lost task 0.0 in stage 18.0 (TID 13) (spark-jupyter executor driver): org.apache.spark.SparkException: Parquet column cannot be converted in file hdfs://namenode:9000/datalake/raw/Purchasing/PurchaseOrderHeader/order_year=2014/order_month=7/part-00000-ed54d3e5-bd61-4a49-9e92-5b0ca45017fd.c000.snappy.parquet. Column: [SubTotal], Expected: decimal(18,2), Found: FIXED_LEN_BYTE_ARRAY.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.unsupportedSchemaColumnConvertError(QueryExecutionErrors.scala:854)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.nextIterator(FileScanRDD.scala:287)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:129)
	at org.apache.spark.sql.execution.FileSourceScanExec$$anon$1.hasNext(DataSourceScanExec.scala:593)
	a

## 3. Extract — Filter & Join

**⚠️ Terapkan filter ShipMethod di sini!**

In [10]:
# Filter hanya ShipMethod CARGO TRANSPORT 5 dan OVERNIGHT J-FAST
VALID_METHODS = ['CARGO TRANSPORT 5', 'OVERNIGHT J-FAST']

# Step 1: Filter ShipMethod
df_methods_filtered = df_shipmethod.filter(F.col('Name').isin(VALID_METHODS))

# Step 2: Join PurchaseOrderHeader dengan ShipMethod yang sudah difilter
# (ini otomatis memfilter PO berdasarkan ShipMethod)
df_po_filtered = df_po_header.join(df_methods_filtered, df_po_header.ShipMethodID == df_methods_filtered.ShipMethodID, 'inner') \
    .select(df_po_header['*'], df_methods_filtered.Name.alias('ship_method'))

print(f'Total PO setelah filter: {df_po_filtered.count():,}')
df_po_filtered.groupBy('ship_method').count().show()

# Step 3: Join dengan PO Detail dan Product
df_enriched = df_po_filtered.alias('h')
df_enriched = df_enriched.join(df_po_detail.alias('d'), F.col('h.PurchaseOrderID') == F.col('d.PurchaseOrderID')) \
    .join(df_vendor.alias('v'), F.col('h.VendorID') == F.col('v.BusinessEntityID')) \
    .join(df_product.alias('p'), F.col('d.ProductID') == F.col('p.ProductID')) \
    .select(
        F.col('h.PurchaseOrderID'),
        F.col('h.OrderDate'),
        F.col('h.ShipDate'),
        F.col('h.DueDate'),
        F.col('h.VendorID').alias('vendor_id'),
        F.col('v.Name').alias('vendor_name'),
        F.col('v.CreditRating').alias('credit_rating'),
        F.col('h.ship_method'),
        F.col('d.UnitPrice'),
        F.col('p.StandardCost')
    )

# Step 4: Tambahkan kolom kalkulasi:
df_enriched = df_enriched.withColumn('is_on_time', F.when((F.col('ShipDate').isNotNull()) & (F.col('ShipDate') <= F.col('DueDate')), 1).otherwise(0)) \
    .withColumn('lead_time_days', F.when(F.col('ShipDate').isNotNull(), F.datediff(F.col('ShipDate'), F.col('OrderDate'))).otherwise(None)) \
    .withColumn('price_variance', F.col('UnitPrice') - F.col('StandardCost'))

print(f'Total rows enriched: {df_enriched.count():,}')
df_enriched.printSchema()

Total PO setelah filter: 2,608
+-----------------+-----+
|      ship_method|count|
+-----------------+-----+
|CARGO TRANSPORT 5| 1523|
| OVERNIGHT J-FAST| 1085|
+-----------------+-----+

Total rows enriched: 6,210
root
 |-- PurchaseOrderID: integer (nullable = true)
 |-- OrderDate: timestamp (nullable = true)
 |-- ShipDate: timestamp (nullable = true)
 |-- DueDate: timestamp (nullable = true)
 |-- vendor_id: integer (nullable = true)
 |-- vendor_name: string (nullable = true)
 |-- credit_rating: short (nullable = true)
 |-- ship_method: string (nullable = true)
 |-- UnitPrice: decimal(19,4) (nullable = true)
 |-- StandardCost: decimal(19,4) (nullable = true)
 |-- is_on_time: integer (nullable = false)
 |-- lead_time_days: integer (nullable = true)
 |-- price_variance: decimal(20,4) (nullable = true)



## 4. Transform — Vendor Summary per ShipMethod

**Grain: 1 baris per (vendor, ship_method)**

In [11]:
# Hitung agregasi per (vendor, ship_method)
df_vendor_summary = df_enriched.groupBy('vendor_id', 'vendor_name', 'credit_rating', 'ship_method').agg(
    F.countDistinct('PurchaseOrderID').alias('total_orders'),
    F.countDistinct(F.when(F.col('is_on_time') == 1, F.col('PurchaseOrderID'))).alias('on_time_orders'),
    F.round((F.countDistinct(F.when(F.col('is_on_time') == 1, F.col('PurchaseOrderID')))/F.countDistinct('PurchaseOrderID'))*100, 2).alias('on_time_rate'),
    F.round(F.avg('lead_time_days'), 1).alias('avg_lead_time_days'),
    F.round(F.avg('price_variance'), 2).alias('avg_price_variance'),
)

# Hitung VendorScore
df_vendor_summary = df_vendor_summary.withColumn(
    'vendor_score',
    F.round(
        (F.col('on_time_rate') * F.lit(0.6)) + ((F.lit(100) - F.abs(F.col('avg_price_variance'))) * F.lit(0.4)),
        2)
)

print(f'Total vendor summary rows: {df_vendor_summary.count():,}')
df_vendor_summary.orderBy(F.desc('vendor_score')).show(20, truncate=False)

Total vendor summary rows: 77
+---------+--------------------------+-------------+-----------------+------------+--------------+------------+------------------+------------------+------------+
|vendor_id|vendor_name               |credit_rating|ship_method      |total_orders|on_time_orders|on_time_rate|avg_lead_time_days|avg_price_variance|vendor_score|
+---------+--------------------------+-------------+-----------------+------------+--------------+------------+------------------+------------------+------------+
|1620     |Lakewood Bicycle          |1            |CARGO TRANSPORT 5|51          |0             |0.0         |9.0               |10.38             |35.85       |
|1568     |Custom Frames, Inc.       |2            |CARGO TRANSPORT 5|1           |0             |0.0         |9.0               |10.47             |35.81       |
|1608     |Sport Playground          |1            |CARGO TRANSPORT 5|50          |0             |0.0         |9.0               |10.82             |35.67 

## 5. Window Functions — Ranking Vendor per ShipMethod

In [12]:
# Rank vendor per ship_method berdasarkan vendor_score
win_by_method = Window.partitionBy('ship_method').orderBy(F.desc('vendor_score'))

df_ranked = df_vendor_summary \
    .withColumn('rank_in_method', F.rank().over(win_by_method)) \
    .withColumn('score_pct_rank', F.percent_rank().over(win_by_method))

print('=== Top 5 Vendor per ShipMethod ===')
df_ranked.filter(F.col('rank_in_method') <= 5) \
         .orderBy('ship_method', 'rank_in_method') \
         .show(20, truncate=False)

# Overall rank (tanpa partisi ship_method)
# Aggregate: kalau satu vendor muncul di dua ship method, ambil avg vendor_score
df_overall = df_vendor_summary.groupBy('vendor_id', 'vendor_name').agg(
    F.round(F.avg('vendor_score'), 2).alias('avg_vendor_score')
)
win_overall = Window.orderBy(F.desc('avg_vendor_score'))
df_overall_ranked = df_overall.withColumn('overall_rank', F.rank().over(win_overall))

print('\n=== Overall Vendor Ranking ===')
df_overall_ranked.show(20, truncate=False)

=== Top 5 Vendor per ShipMethod ===
+---------+---------------------+-------------+-----------------+------------+--------------+------------+------------------+------------------+------------+--------------+--------------+
|vendor_id|vendor_name          |credit_rating|ship_method      |total_orders|on_time_orders|on_time_rate|avg_lead_time_days|avg_price_variance|vendor_score|rank_in_method|score_pct_rank|
+---------+---------------------+-------------+-----------------+------------+--------------+------------+------------------+------------------+------------+--------------+--------------+
|1620     |Lakewood Bicycle     |1            |CARGO TRANSPORT 5|51          |0             |0.0         |9.0               |10.38             |35.85       |1             |0.0           |
|1568     |Custom Frames, Inc.  |2            |CARGO TRANSPORT 5|1           |0             |0.0         |9.0               |10.47             |35.81       |2             |0.2           |
|1608     |Sport Playgro

## 6. Load — Simpan ke Hive Curated

In [ ]:
# Rename kolom agar sesuai target schema, tambah load_timestamp
spark.sql('CREATE DATABASE IF NOT EXISTS adventureworks_curated')

df_final = df_vendor_summary \
    .withColumn('load_timestamp', F.current_timestamp())

# Simpan ke adventureworks_curated.fact_vendor_performance
# mode: overwrite, saveAsTable
df_final.write.mode('overwrite').saveAsTable('adventureworks_curated.fact_vendor_performance')

print('=== Verifikasi ===')
df_verify = spark.table('adventureworks_curated.fact_vendor_performance')
print(f'Total rows: {df_verify.count():,}')
df_verify.show(10, truncate=False)

## 7. Analytic Queries

**Wajib: minimal 3 query yang menjawab business question**

In [ ]:
# ── Query 1 (WAJIB): Ranking vendor berdasarkan VendorScore ────────────────
print('=== Query 1: Vendor Ranking by VendorScore ===')
spark.sql("""
    SELECT vendor_name, ship_method, on_time_rate, avg_price_variance, vendor_score
    FROM adventureworks_curated.fact_vendor_performance
    ORDER BY vendor_score DESC
    LIMIT 100
""").show(truncate=False)

In [ ]:
# ── Query 2 (WAJIB): Perbandingan OnTimeRate per ShipMethod ────────────────
print('=== Query 2: OnTimeRate per ShipMethod ===')
spark.sql("""
    SELECT ship_method,
           ROUND(AVG(on_time_rate),2) AS avg_on_time_rate,
           COUNT(DISTINCT vendor_id) AS total_vendors,
           SUM(total_orders) AS total_orders
    FROM adventureworks_curated.fact_vendor_performance
    GROUP BY ship_method
    ORDER BY avg_on_time_rate DESC
""").show(truncate=False)

In [ ]:
# ── Query 3 (WAJIB): Vendor dengan AvgPriceVariance negatif ───────────────
print('=== Query 3: Vendor dengan Harga Lebih Murah dari StandardCost ===')
spark.sql("""
    SELECT vendor_name, ship_method, avg_price_variance, on_time_rate, vendor_score
    FROM adventureworks_curated.fact_vendor_performance
    WHERE avg_price_variance < 0
    ORDER BY avg_price_variance ASC
""").show(truncate=False)

In [ ]:
# ── Query 4 (BONUS): Korelasi CreditRating dan OnTimeRate ──────────────────
print('=== Query 4 (Bonus): CreditRating vs OnTimeRate ===')
spark.sql("""
    SELECT credit_rating,
           ROUND(AVG(on_time_rate),2) AS avg_on_time_rate,
           ROUND(AVG(vendor_score),2) AS avg_vendor_score,
           COUNT(*) AS count_vendors
    FROM adventureworks_curated.fact_vendor_performance
    GROUP BY credit_rating
    ORDER BY credit_rating DESC
""").show(truncate=False)

In [ ]:
# ── Query 5 (BONUS): Top 3 vendor per ShipMethod berdasarkan TotalOrders ───
print('=== Query 5 (Bonus): Top 3 Vendor per ShipMethod by TotalOrders ===')
spark.sql("""
    SELECT vendor_name, ship_method, total_orders, rnk FROM (
        SELECT vendor_name, ship_method, total_orders,
               RANK() OVER (PARTITION BY ship_method ORDER BY total_orders DESC) AS rnk
        FROM adventureworks_curated.fact_vendor_performance
    ) t
    WHERE rnk <= 3
    ORDER BY ship_method, rnk
""").show(truncate=False)

In [ ]:
spark.stop()
print('Done!')